# Загрузка библиотек #

In [2]:
import pandas as pd
from datasets import load_dataset
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score
from sklearn.model_selection import train_test_split
import random
import torch
from transformers import (
    AutoConfig,
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)
from sklearn.preprocessing import LabelEncoder
import json

# Загружаем RuTweetCorp #

In [7]:
df_pos = pd.read_csv('drive/MyDrive/hateeggea/positive.csv', encoding='utf-8')
df_neg = pd.read_csv('drive/MyDrive/hateeggea/negative.csv', encoding='utf-8')


df_pos_clean = pd.DataFrame({
    'text': df_pos['ttext'],
    'emotion': 'joy'
})

df_neg_clean = pd.DataFrame({
    'text': df_neg['ttext'],
    'emotion': 'sadness'
})


rutweet_combined = pd.concat([df_pos_clean, df_neg_clean], ignore_index=True)
print(rutweet_combined.head())

                                                text emotion
0  @first_timee хоть я и школота, но поверь, у на...     joy
1  Да, все-таки он немного похож на него. Но мой ...     joy
2  RT @KatiaCheh: Ну ты идиотка) я испугалась за ...     joy
3  RT @digger2912: "Кто то в углу сидит и погибае...     joy
4  @irina_dyshkant Вот что значит страшилка :D\nН...     joy


# Загружаем CEDR #

In [8]:
cedr = load_dataset("cedr", "main")

def map_emotion(label):
    mapping = {
        0: "joy",
        1: "sadness",
        2: "surprise",
        3: "fear",
        4: "anger"
    }

    if isinstance(label, list):
        label = label[0] if label else None
    return mapping.get(label, None)


cedr_train = pd.DataFrame(cedr["train"])
cedr_test = pd.DataFrame(cedr["test"])

cedr_train["emotion"] = cedr_train["labels"].apply(map_emotion)
cedr_test["emotion"] = cedr_test["labels"].apply(map_emotion)

cedr_train_clean = cedr_train[["text", "emotion"]].dropna()
cedr_test_clean = cedr_test[["text", "emotion"]].dropna()

print(f"CEDR train: {len(cedr_train_clean)}")
print(cedr_train_clean["emotion"].value_counts())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

main/train-00000-of-00001.parquet:   0%|          | 0.00/757k [00:00<?, ?B/s]

main/test-00000-of-00001.parquet:   0%|          | 0.00/188k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/7528 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1882 [00:00<?, ? examples/s]

CEDR train: 4485
emotion
joy         1548
sadness     1403
surprise     574
fear         570
anger        390
Name: count, dtype: int64


# Объединяем датасеты #

In [9]:
combined = pd.concat([
    rutweet_combined,
    cedr_train_clean,
    cedr_test_clean
], ignore_index=True)

print(f"Общий размер: {len(combined)}")
print(combined["emotion"].value_counts())

Общий размер: 232467
emotion
joy         116806
sadness     113704
surprise       739
fear           706
anger          512
Name: count, dtype: int64


# Обучаем бейзлайн модель #

In [10]:
X = combined['text'].astype(str).tolist()
y = combined['emotion'].tolist()


X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f"Train size: {len(X_train)}, Test size: {len(X_test)}")

# TF-IDF векторизация
tfidf = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

# Обучение с балансировкой классов
baseline = LogisticRegression(max_iter=1000, class_weight='balanced')
baseline.fit(X_train_tfidf, y_train)

y_pred = baseline.predict(X_test_tfidf)

print(f"Accuracy: {accuracy_score(y_test, y_pred):.4f}")
print(f"F1-score (weighted): {f1_score(y_test, y_pred, average='weighted'):.4f}")
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

Train size: 185973, Test size: 46494
Accuracy: 0.7130
F1-score (weighted): 0.7260

Classification Report:
              precision    recall  f1-score   support

       anger       0.04      0.33      0.08       102
        fear       0.11      0.61      0.19       141
         joy       0.75      0.73      0.74     23362
     sadness       0.75      0.70      0.72     22741
    surprise       0.09      0.51      0.15       148

    accuracy                           0.71     46494
   macro avg       0.35      0.58      0.37     46494
weighted avg       0.74      0.71      0.73     46494



# Создаем дополнительные примеры в наши редкие классы(anger, fear, surprise) #

In [11]:
def augment_text(text, n_variations=2):
    variations = [text]

    # 1. Удаление пунктуации в конце
    if text.endswith(('.', '!', '?')):
        variations.append(text[:-1])

    # 2. Добавление восклицательного знака для эмоциональных текстов
    if '!' not in text:
        variations.append(text + '!')

    # 3. Повторение последнего слова для эмфазы
    words = text.split()
    if len(words) > 1:
        last_word = words[-1]
        variations.append(text + ' ' + last_word)

    return list(set(variations))[:n_variations]


augmented_data = []

rare_classes = ['anger', 'fear', 'surprise']

for emotion in rare_classes:
    class_samples = combined[combined['emotion'] == emotion]
    print(f"Оригинальных {emotion}: {len(class_samples)}")

    for _, row in class_samples.iterrows():
        text = row['text']
        augmented_texts = augment_text(text, n_variations=2)

        for aug_text in augmented_texts:
            if aug_text != text:
                augmented_data.append({
                    'text': aug_text,
                    'emotion': emotion
                })

augmented_df = pd.DataFrame(augmented_data)

combined_augmented = pd.concat([combined, augmented_df], ignore_index=True)

print(f"\nРазмер после аугментации: {len(combined_augmented)}")
print(combined_augmented['emotion'].value_counts())

Оригинальных anger: 512
Оригинальных fear: 706
Оригинальных surprise: 739

Размер после аугментации: 235263
emotion
joy         116806
sadness     113704
surprise      1829
fear          1703
anger         1221
Name: count, dtype: int64


# Обучаем ruBERT-tiny #

In [14]:
label_encoder = LabelEncoder()
combined_augmented['label_id'] = label_encoder.fit_transform(combined_augmented['emotion'])

class_counts = combined_augmented['emotion'].value_counts()
class_weights = {}
for label in label_encoder.classes_:
    class_weights[label] = 1.0 / class_counts[label]

print("Веса классов:")
for label, weight in class_weights.items():
    print(f"  {label}: {weight:.6f}")

class_weights_list = [class_weights[label] for label in label_encoder.classes_]
class_weights_tensor = torch.tensor(class_weights_list, dtype=torch.float)

print(f"\nТензор весов: {class_weights_tensor}")

train_texts, val_texts, train_labels, val_labels = train_test_split(
    combined_augmented['text'].tolist(),
    combined_augmented['label_id'].tolist(),
    test_size=0.15,
    random_state=42,
    stratify=combined_augmented['label_id']
)

print(f"Train: {len(train_texts)}, Val: {len(val_texts)}")

model_name = "cointegrated/rubert-tiny2"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(label_encoder.classes_)
)

original_loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

def compute_loss_with_weights(self, outputs, labels):
    logits = outputs.logits
    return original_loss_fn(logits, labels)

model.compute_loss = compute_loss_with_weights.__get__(model, AutoModelForSequenceClassification)

def tokenize_function(texts):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )

train_encodings = tokenize_function(train_texts)
val_encodings = tokenize_function(val_texts)

class EmotionDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = EmotionDataset(train_encodings, train_labels)
val_dataset = EmotionDataset(val_encodings, val_labels)

def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    acc = accuracy_score(labels, predictions)
    f1 = f1_score(labels, predictions, average='weighted')
    return {"accuracy": acc, "f1": f1}

training_args = TrainingArguments(
    output_dir="./rubert_emoji_results",
    num_train_epochs=5,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_ratio=0.1,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

model.save_pretrained("drive/MyDrive/hateeggea/emoji_rubert_model")
tokenizer.save_pretrained("drive/MyDrive/hateeggea/emoji_rubert_model")

with open("drive/MyDrive/hateeggea/emoji_rubert_model/label_mapping.json", "w") as f:
    json.dump(label_encoder.classes_.tolist(), f)

print("Сохранено:")

Веса классов:
  anger: 0.000819
  fear: 0.000587
  joy: 0.000009
  sadness: 0.000009
  surprise: 0.000547

Тензор весов: tensor([8.1900e-04, 5.8720e-04, 8.5612e-06, 8.7948e-06, 5.4675e-04])
Train: 199973, Val: 35290


Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cointegrated/rubert-tiny2
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider trai

Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.036046,0.025054,0.992547,0.992471
2,0.015383,0.018295,0.994446,0.994455
3,0.003748,0.019505,0.995551,0.995569
4,0.006261,0.018572,0.995976,0.995986
5,0.004425,0.018596,0.996175,0.996235


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta', 'bert.encoder.layer.0.attention.output.LayerNorm.gamma', 'bert.

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Сохранено:


In [3]:
import os
import torch.nn as nn

OLD_MODEL_PATH = '/content/drive/MyDrive/hateeggea/emoji_rubert_model'
NEW_MODEL_PATH = '/content/drive/MyDrive/hateeggea/emoji_rubert_model_fixed_v2'

os.makedirs(NEW_MODEL_PATH, exist_ok=True)

config = AutoConfig.from_pretrained(OLD_MODEL_PATH)
config.num_labels = 5

model = AutoModelForSequenceClassification.from_config(config)

weights_loaded = False
possible_weight_files = ['pytorch_model.bin', 'model.safetensors', 'adapter_model.bin']

for weight_file in possible_weight_files:
    weight_path = os.path.join(OLD_MODEL_PATH, weight_file)
    if os.path.exists(weight_path):
        print(f"   Найден файл весов: {weight_file}")
        try:
            if weight_file.endswith('.safetensors'):
                from safetensors.torch import load_file
                state_dict = load_file(weight_path)
            else:
                state_dict = torch.load(weight_path, map_location='cpu')

            missing_keys, unexpected_keys = model.load_state_dict(state_dict, strict=False)
            print(f"Веса успешно загружены из {weight_file}")
            if missing_keys:
                print(f"Отсутствующие ключи: {missing_keys[:3]}...")
            if unexpected_keys:
                print(f"Неожиданные ключи: {unexpected_keys[:3]}...")
            weights_loaded = True
            break
        except Exception as e:
            print(f"Не удалось загрузить {weight_file}: {e}")

if not weights_loaded:
    print("Не найден файл с весами в папке модели!")
else:
    model.save_pretrained(NEW_MODEL_PATH, safe_serialization=False)

    tokenizer = AutoTokenizer.from_pretrained(OLD_MODEL_PATH)
    tokenizer.save_pretrained(NEW_MODEL_PATH)

    label_map_path = os.path.join(OLD_MODEL_PATH, 'label_mapping.json')
    if os.path.exists(label_map_path):
        import shutil
        shutil.copy(label_map_path, NEW_MODEL_PATH)
        print("Файл label_mapping.json скопирован.")
    else:
        print("Файл label_mapping.json не найден, создаём вручную...")
        class_names = ['anger', 'fear', 'joy', 'sadness', 'surprise']
        with open(os.path.join(NEW_MODEL_PATH, 'label_mapping.json'), 'w') as f:
            json.dump(class_names, f)
        print("Файл label_mapping.json создан.")

    print(f"\nМОДЕЛЬ УСПЕШНО ПЕРЕСОХРАНЕНА!")
    print(f"Новый путь: {NEW_MODEL_PATH}")

   Найден файл весов: model.safetensors
Веса успешно загружены из model.safetensors
Отсутствующие ключи: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight']...
Неожиданные ключи: ['bert.embeddings.LayerNorm.beta', 'bert.embeddings.LayerNorm.gamma', 'bert.encoder.layer.0.attention.output.LayerNorm.beta']...


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Файл label_mapping.json скопирован.

МОДЕЛЬ УСПЕШНО ПЕРЕСОХРАНЕНА!
Новый путь: /content/drive/MyDrive/hateeggea/emoji_rubert_model_fixed_v2


In [7]:
import telebot

MODEL_PATH = '/content/drive/MyDrive/hateeggea/emoji_rubert_model_fixed_v2'
BOT_TOKEN = '8813101812:AAHIH-KqHiLJsoyqfZHsUnV66vBozfrhF8k'

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

with open(f'{MODEL_PATH}/label_mapping.json', 'r') as f:
    class_names = json.load(f)

label_encoder = LabelEncoder()
label_encoder.classes_ = np.array(class_names)

EMOJI_MAP = {
    'anger': ['😠', '😡', '🤬'],
    'fear': ['😨', '😰', '😱'],
    'joy': ['😊', '😂', '😄', '🎉', '😍', '❤️'],
    'sadness': ['😢', '😔', '😭', '💔', '🥺'],
    'surprise': ['😲', '😱', '😯', '🤯', '😳']
}

def get_emoji(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
        pred_id = torch.argmax(outputs.logits, dim=-1).item()

    emotion = label_encoder.inverse_transform([pred_id])[0]
    return random.choice(EMOJI_MAP.get(emotion, ['❓']))

bot = telebot.TeleBot(BOT_TOKEN)

@bot.message_handler(func=lambda message: True)
def handle_all_messages(message):
    if message.text and not message.text.startswith('/'):
        try:
            emoji = get_emoji(message.text)
            bot.reply_to(message, f"{message.text} {emoji}")
        except:
            bot.reply_to(message, f"{message.text} ❓")

print("Бот запущен!")
bot.infinity_polling()

Loading weights:   0%|          | 0/57 [00:00<?, ?it/s]

Бот запущен!


2026-05-22 23:31:49,983 (__init__.py:1134 MainThread) ERROR - TeleBot: "Infinity polling: polling exited"
ERROR:TeleBot:Infinity polling: polling exited
2026-05-22 23:31:50,001 (__init__.py:1136 MainThread) ERROR - TeleBot: "Break infinity polling"
ERROR:TeleBot:Break infinity polling
